In [6]:
#Because sample size is so small I will use Gemini to analyze the sentiment
import os
from pathlib import Path
import sys
from google import genai
from google.genai import types

parent_dir = Path.cwd().parent
sys.path.insert(0, str(parent_dir))
from config import GEMINI_API_KEY
#Initialize the Gemini client
client = genai.Client(api_key=GEMINI_API_KEY)




In [21]:
import re
def evaluate_sentiment(text: str) -> float:
    # Create a request for the Gemini API
    prompt = f"""
        This text is pulled from the nascar subreddit. It may or may not discuss sentiment toward one of these sponsors: FedEx, Cheddar's Scratch Kitchen, Love's Travel Stops, Busch Light, or Castrol.

        Analyze the sentiment expressed toward the sponsor in the text below. Respond with ONLY a single number between -1 and 1, where -1 is very negative, 0 is neutral, and 1 is very positive. Do not include any explanation, words, or punctuation — output the number only.

        Text: {text}
        """
    response = client.models.generate_content(
        model="gemini-3.1-flash-lite",
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=0.0,
            max_output_tokens=10,
            thinking_config=types.ThinkingConfig(thinking_budget=0)
        )
    )
    assert response.text is not None, "No response from Gemini API"
    raw = response.text.strip()
    assert raw != "", "Empty response from Gemini API"
    match = re.search(r"-?\d+\.?\d*", raw)
    assert match, f"Could not parse a number from response: {raw!r}"
    return float(match.group())
#Test Use
test_text = "FedEx driver so buns! I can't watch this crap"
result = evaluate_sentiment(test_text)
print(f"Sentiment score for test text: {result}")

Sentiment score for test text: -1.0


In [15]:
import pandas as pd
df_FedEx = pd.read_csv('data/raw/FedEx.csv')
df_Cheddars = pd.read_csv('data/raw/Cheddar\'s Scratch Kitchen.csv')
df_Loves = pd.read_csv('data/raw/Love\'s Travel Stops.csv')
df_Busch = pd.read_csv('data/raw/Busch Light.csv')
df_Castrol = pd.read_csv('data/raw/Castrol.csv')

In [24]:
import time

dfs = [df_FedEx, df_Cheddars, df_Loves, df_Busch, df_Castrol]
date_sentiment = []
for df in dfs:
    df = list(zip(df['title'], df['published']))
    for title, date in df:
        sentiment_score = evaluate_sentiment(title)
        date_sentiment.append((date, sentiment_score))
        time.sleep(3)  # Pause for 3 seconds to avoid hitting rate limits
        print(f"Title: {title}, Date: {date}, Sentiment Score: {sentiment_score}")
        

final_df = pd.DataFrame(date_sentiment, columns=['date', 'sentiment_score'])


Title: After winning Michigan,Denny Hamlin honors kyle busch with a black flag with the number 18 and Kyle's name in cursive, Date: 2026-06-07 23:27:31+00:00, Sentiment Score: 0.0
Title: Denny Hamlin confirms on the Prime post race show that Brent Crews will take over the 11 after he retires at the end of 2027., Date: 2026-06-07 23:56:55+00:00, Sentiment Score: 0.0
Title: [Gluck] Denny Hamlin says he put plans in motion starting Monday morning to come up with the tribute flag and logo for Kyle Busch in case he won Michigan. Denny says they were able to collaborate with RCR to get the stylized 8 and combine it with the JGR 18 (the 1 was used and is in black)., Date: 2026-06-08 00:39:04+00:00, Sentiment Score: 0.0
Title: Denny Hamlin driving a 2013 Jordan Kyle Busch Motorsports truck at fall Martinsville, Date: 2026-06-08 03:13:38+00:00, Sentiment Score: 0.0
Title: The Day After the Races - June 8, 2026, Date: 2026-06-08 11:00:07+00:00, Sentiment Score: 0.0
Title: Denny Hamlin will have 

KeyboardInterrupt: 